# Honey Chain — 7-Day Honey Yield Forecasting

This notebook trains and evaluates the Honey Chain prototype Random Forest model. **The supplied honey-production data is synthetic; this is a proof-of-concept, not a field-validated model.**


In [ ]:
!pip -q install pandas numpy scikit-learn joblib xgboost

from google.colab import files
import pandas as pd, numpy as np, io, os

print("Upload these 3 CSVs:")
print("1. Honey_Production_Dataset_2023.csv")
print("2. Honey_Production_Dataset_2024_Ideal.csv")
print("3. Honey_Production_Dataset_2024_NonIdeal.csv")
uploaded = files.upload()


In [ ]:
# Save uploaded files and import the reusable training pipeline
for name, data in uploaded.items():
    open(name, 'wb').write(data)

# If you cloned the GitHub repository, this import will work after %cd into the repo.
# For a standalone Colab, the complete implementation is reproduced below.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

DATE='Date'; TARGET='Honey Weight (kg)'; EXTRACTION='Extract Honey'
ENV={'Environmental Temperature (°C)':'env_temp','Relative Humidity (%)':'env_humidity','Hive Temperature (°C)':'hive_temp','Hive Humidity (%)':'hive_humidity','Wind Speed (km/h)':'wind_speed'}

def load_df(path):
    d=pd.read_csv(path); d[DATE]=pd.to_datetime(d[DATE]);
    for c in list(ENV)+[TARGET]: d[c]=pd.to_numeric(d[c], errors='coerce')
    d[EXTRACTION]=d[EXTRACTION].astype(str).str.lower().isin(['true','1','yes'])
    return d.sort_values(DATE).reset_index(drop=True)

def make_features(df, target=True):
    d=df.copy().sort_values(DATE).reset_index(drop=True)
    last=None; ages=[]
    for _,r in d.iterrows():
        if r[EXTRACTION]: last=r[DATE]; ages.append(0.0)
        elif last is None: ages.append(999.0)
        else: ages.append((r[DATE]-last).days)
    d['days_since_last_extraction']=ages
    for lag in [1,3,7,14]: d[f'honey_lag{lag}']=d[TARGET].shift(lag)
    d['honey_roll_mean_7']=d[TARGET].shift(1).rolling(7,min_periods=1).mean()
    d['honey_roll_std_7']=d[TARGET].shift(1).rolling(7,min_periods=2).std().fillna(0)
    d['honey_roll_mean_14']=d[TARGET].shift(1).rolling(14,min_periods=1).mean()
    d['honey_diff_1']=d[TARGET].shift(1).diff()
    feats=['honey_lag1','honey_lag3','honey_lag7','honey_lag14','honey_roll_mean_7','honey_roll_std_7','honey_roll_mean_14','honey_diff_1','days_since_last_extraction']
    for src,short in ENV.items():
        d[short]=d[src]; d[f'{short}_roll7']=d[src].shift(1).rolling(7,min_periods=1).mean(); feats += [short,f'{short}_roll7']
    d['doy_sin']=np.sin(2*np.pi*d[DATE].dt.dayofyear/365.25); d['doy_cos']=np.cos(2*np.pi*d[DATE].dt.dayofyear/365.25); d['month']=d[DATE].dt.month
    feats += ['doy_sin','doy_cos','month']
    if target:
        d['target_7d']=d[TARGET].shift(-7); d=d.dropna(subset=feats+['target_7d'])
    else: d=d.dropna(subset=feats)
    return d.reset_index(drop=True),feats

def score(y,p):
    return {'MAE':mean_absolute_error(y,p),'RMSE':mean_squared_error(y,p)**0.5,'R2':r2_score(y,p)}

train_df=pd.concat([load_df('Honey_Production_Dataset_2023.csv'),load_df('Honey_Production_Dataset_2024_Ideal.csv')],ignore_index=True).sort_values(DATE)
all_df,features=make_features(train_df)
split=int(len(all_df)*.8); tr=all_df.iloc[:split]; te=all_df.iloc[split:]
models={
 'Linear Regression':LinearRegression(),
 'Random Forest':RandomForestRegressor(n_estimators=500,random_state=42,n_jobs=-1),
 'XGBoost':XGBRegressor(n_estimators=500,max_depth=5,learning_rate=.03,subsample=.9,colsample_bytree=.9,objective='reg:squarederror',random_state=42,n_jobs=-1)
}
results={'Persistence':score(te['target_7d'],te[TARGET])}
for name,m in models.items():
    m.fit(tr[features],tr['target_7d']); results[name]=score(te['target_7d'],m.predict(te[features]))
print(pd.DataFrame(results).T.round(3))

rf=models['Random Forest']; print('\nTop features:')
print(pd.Series(rf.feature_importances_,index=features).sort_values(ascending=False).head(10).round(4))


In [ ]:
# Out-of-domain test on non-ideal 2024
ood,_=make_features(load_df('Honey_Production_Dataset_2024_NonIdeal.csv'))
print({name:score(ood['target_7d'],model.predict(ood[features])) for name,model in models.items()})


In [ ]:
# Save the trained Random Forest for the Honey Chain backend
import joblib, json
bundle={'model':rf,'features':features,'horizon_days':7,'target':TARGET,'model_name':'random_forest','note':'Prototype trained on synthetic honey-production data.'}
joblib.dump(bundle,'honey_yield_model.joblib')
print('Saved honey_yield_model.joblib')
